# Paper 39 — Reale (2014): Coronal Loops Implementation / 코로나 루프 구현

**Paper**: Reale, F., "Coronal Loops: Observations and Modeling of Confined Plasma", *Living Rev. Solar Phys.*, **11**, 4 (2014). DOI: 10.12942/lrsp-2014-4

## Overview / 개요

**English**: This notebook reproduces the key quantitative results of Reale (2014): (1) RTV scaling laws T_max ∝ (pL)^(1/3); (2) a hydrostatic loop profile T(s), n(s) solved via a simplified Serio-like model; (3) the Spitzer thermal conductivity vs. T; (4) the Rosner-Tucker-Vaiana piecewise radiative loss function Λ(T); (5) the nanoflare energy power-law dN/dE ∝ E^(−α) with α = 1.5 vs 2.5; (6) the kink-mode oscillation period P = 2L/c_k from coronal seismology; (7) cooling timescales comparison (conductive vs radiative).

**한국어**: 이 노트북은 Reale (2014)의 주요 정량 결과를 재현한다: (1) RTV 스케일링 T_max ∝ (pL)^(1/3); (2) 간소화된 Serio형 모델로 푸는 정역학 루프 단면 T(s), n(s); (3) 온도에 따른 Spitzer 열전도도; (4) Rosner-Tucker-Vaiana 구간별 복사 손실 함수 Λ(T); (5) α = 1.5 vs 2.5의 나노플레어 에너지 멱법칙 dN/dE ∝ E^(−α); (6) 코로나 지진학의 kink 모드 진동 주기 P = 2L/c_k; (7) 냉각 시간 비교 (전도 vs 복사).

In [ ]:
"""Imports and physical constants for coronal loop modeling.

This cell sets up the standard scientific Python stack (NumPy, Matplotlib)
and defines CGS constants used throughout the notebook.
"""
import numpy as np
import matplotlib.pyplot as plt

# CGS physical constants.
K_B = 1.380649e-16        # Boltzmann constant [erg/K]
M_H = 1.6735e-24          # Hydrogen mass [g]
G_SUN = 2.74e4            # Solar surface gravity [cm/s^2]
KAPPA_SP = 9.0e-7         # Spitzer thermal conductivity coefficient (cgs)

plt.rcParams.update({'figure.figsize': (8, 5), 'axes.grid': True, 'grid.alpha': 0.3})
print('Environment ready: NumPy', np.__version__)

## 1. RTV Scaling Law: T_max vs pL / RTV 스케일링: T_max vs pL

**English**: The Rosner-Tucker-Vaiana (1978) scaling for a hydrostatic, uniformly-heated loop is $T_{\max} = 1.4 \times 10^3 (pL)^{1/3}$ K, where p is pressure in dyne cm⁻² and L is loop half-length in cm. We plot T_max against the product pL over the observed coronal range (pL from 10⁸ to 10¹¹ dyne cm⁻¹) and overlay typical warm/hot/flaring loop bands.

**한국어**: Rosner-Tucker-Vaiana (1978) 스케일링은 정역학·균일가열 루프에 대해 $T_{\max} = 1.4 \times 10^3 (pL)^{1/3}$ K 이며, p [dyne cm⁻²], L [cm]. 관측되는 코로나 범위 (pL = 10⁸–10¹¹ dyne cm⁻¹)에 대해 T_max를 그리고 대표 warm/hot/flaring 루프 영역을 덧그린다.

In [ ]:
def rtv_tmax(p_dyne, L_cm):
    """Compute RTV maximum loop temperature.

    Args:
        p_dyne: Loop pressure [dyne cm^-2].
        L_cm: Loop half-length [cm].

    Returns:
        T_max [K] from T_max = 1.4e3 (p*L)^(1/3).
    """
    return 1.4e3 * (p_dyne * L_cm) ** (1.0 / 3.0)


def rtv_heating(p_dyne, L_cm):
    """Compute RTV volumetric heating rate.

    Args:
        p_dyne: Loop pressure [dyne cm^-2].
        L_cm: Loop half-length [cm].

    Returns:
        H [erg cm^-3 s^-1] from H = 3e-3 * p^(7/6) * (L/1e9)^(-5/6).
    """
    L9 = L_cm / 1e9
    return 3.0e-3 * p_dyne ** (7.0 / 6.0) * L9 ** (-5.0 / 6.0)


pL = np.logspace(8, 11, 200)
T_max = 1.4e3 * pL ** (1.0 / 3.0)

fig, ax = plt.subplots()
ax.loglog(pL, T_max / 1e6, 'b-', lw=2.2, label='RTV: $T_{\\max}=1.4\\times 10^3 (pL)^{1/3}$')
ax.axhspan(1, 1.5, color='green', alpha=0.2, label='Warm (1–1.5 MK)')
ax.axhspan(2, 6, color='orange', alpha=0.2, label='Hot (2–6 MK)')
ax.axhspan(10, 30, color='red', alpha=0.15, label='Flaring (>10 MK)')
ax.set_xlabel('pL [dyne cm$^{-1}$]')
ax.set_ylabel('T$_{max}$ [MK]')
ax.set_title('RTV Scaling Law: T$_{max}$ vs pL / RTV 스케일링')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

p_ex, L_ex = 2.0, 5e9
print(f'Example AR loop: p={p_ex} dyne/cm^2, L={L_ex:.1e} cm')
print(f'  T_max = {rtv_tmax(p_ex, L_ex)/1e6:.2f} MK')
print(f'  H     = {rtv_heating(p_ex, L_ex):.2e} erg cm^-3 s^-1')

## 2. Hydrostatic Loop Profile T(s), n(s) / 정역학 루프 단면

**English**: We solve a simplified hydrostatic loop profile: temperature varies along s from transition-region base (T_base ≈ 2×10⁴ K) to apex (T_0) following a Serio-like power-law approximation, and density follows gravitational stratification n(s) via an exponential fall-off using the pressure scale height $H_p = k_B T/(m_H g_\odot)$. We assume a semicircular loop of half-length L so height h(s) = (L/π) sin(πs/L).

**한국어**: 간소화된 정역학 루프 단면을 푼다: 온도는 전이영역 밑면 (T_base ≈ 2×10⁴ K)에서 정점 (T_0)까지 Serio형 멱법칙 근사로 변화하고, 밀도는 압력 스케일 높이 $H_p = k_B T/(m_H g_\odot)$를 이용한 지수적 감쇄로 중력 성층을 따른다. 반원형 루프로 가정하여 높이는 h(s) = (L/π) sin(πs/L).

In [ ]:
def hydrostatic_loop(L, T0, n_base=1e10, T_base=2e4, n_points=200):
    """Compute temperature and density profile along a semicircular hydrostatic loop.

    Args:
        L: Loop half-length [cm].
        T0: Apex temperature [K].
        n_base: Base density at transition-region top [cm^-3].
        T_base: Base temperature [K].
        n_points: Number of grid points from base to apex.

    Returns:
        Tuple (s, h, T, n) arrays along loop.
    """
    s = np.linspace(0, L, n_points)
    h = (L / np.pi) * np.sin(np.pi * s / L)
    L_TR = 0.02 * L
    T = np.where(
        s < L_TR,
        T_base + (T0 - T_base) * (s / L_TR) ** 0.5,
        T0 * (1.0 - 0.15 * (1.0 - np.sin(np.pi * s / L)))
    )
    T_avg = 0.5 * (T + T0)
    H_avg = K_B * T_avg / (M_H * G_SUN)
    p_base = n_base * K_B * T_base
    p = p_base * np.exp(-h / H_avg)
    n = p / (K_B * T)
    return s, h, T, n


L_loop = 1e10
s_ar, h_ar, T_ar, n_ar = hydrostatic_loop(L_loop, T0=3e6, n_base=1e10)
s_em, h_em, T_em, n_em = hydrostatic_loop(L_loop, T0=1e6, n_base=1e8)

fig, axs = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
axs[0].semilogy(s_ar / 1e9, T_ar, 'r-', lw=2, label='AR (T$_0$=3 MK)')
axs[0].semilogy(s_em / 1e9, T_em, 'g--', lw=2, label='Empty (T$_0$=1 MK)')
axs[0].set_ylabel('T [K]')
axs[0].set_title('Hydrostatic Loop Profile / 정역학 루프 단면')
axs[0].legend()

axs[1].semilogy(s_ar / 1e9, n_ar, 'r-', lw=2, label='AR')
axs[1].semilogy(s_em / 1e9, n_em, 'g--', lw=2, label='Empty')
axs[1].set_ylabel('n [cm$^{-3}$]')
axs[1].legend()

axs[2].semilogy(s_ar / 1e9, n_ar * K_B * T_ar, 'r-', lw=2, label='AR')
axs[2].semilogy(s_em / 1e9, n_em * K_B * T_em, 'g--', lw=2, label='Empty')
axs[2].set_ylabel('Pressure [dyne cm$^{-2}$]')
axs[2].set_xlabel('s along loop [$10^9$ cm]')
axs[2].legend()
plt.tight_layout()
plt.show()

print(f'AR loop apex: T={T_ar[-1]:.2e} K, n={n_ar[-1]:.2e} cm^-3')
print(f'Empty apex:   T={T_em[-1]:.2e} K, n={n_em[-1]:.2e} cm^-3')

## 3. Spitzer Thermal Conductivity / Spitzer 열전도도

**English**: Spitzer thermal conductivity for fully-ionized plasma is $\kappa(T) = \kappa_0 T^{5/2}$ with $\kappa_0 \approx 9 \times 10^{-7}$ erg cm⁻¹ s⁻¹ K⁻⁷ᐟ². The T^{5/2} dependence makes conduction overwhelmingly dominant at coronal temperatures.

**한국어**: 완전 이온화 플라스마의 Spitzer 열전도도는 $\kappa(T) = \kappa_0 T^{5/2}$, $\kappa_0 \approx 9 \times 10^{-7}$ erg cm⁻¹ s⁻¹ K⁻⁷ᐟ². T^{5/2} 의존성이 코로나 온도에서 전도를 압도적으로 지배적이게 만든다.

In [ ]:
def spitzer_conductivity(T):
    """Spitzer thermal conductivity for fully-ionized plasma.

    Args:
        T: Temperature [K].

    Returns:
        kappa(T) [erg cm^-1 s^-1 K^-1] = kappa_0 * T^(5/2).
    """
    return KAPPA_SP * T ** 2.5


T_range = np.logspace(4, 8, 300)
kappa = spitzer_conductivity(T_range)

fig, ax = plt.subplots()
ax.loglog(T_range, kappa, 'b-', lw=2.2)
ax.axvspan(1e6, 1e7, alpha=0.15, color='orange', label='Coronal range (1–10 MK)')
ax.set_xlabel('T [K]')
ax.set_ylabel('$\\kappa$ [erg cm$^{-1}$ s$^{-1}$ K$^{-1}$]')
ax.set_title('Spitzer Thermal Conductivity $\\kappa = 9\\times10^{-7}\\, T^{5/2}$')
ax.legend()
plt.tight_layout()
plt.show()

for T_ex in [1e5, 1e6, 3e6, 1e7]:
    print(f'T = {T_ex:.0e} K -> kappa = {spitzer_conductivity(T_ex):.3e} erg/(cm s K)')

## 4. Radiative Loss Function Λ(T) / 복사 손실 함수

**English**: The optically-thin radiative loss function $\Lambda(T)$ (erg cm³ s⁻¹) for coronal abundances is approximated by Rosner, Tucker & Vaiana (1978) as a piecewise power law. It rises toward T ~ 10^5.3 K (chromosphere/TR peak), plateaus across 10⁶–10⁷ K, then decreases toward high T (bremsstrahlung regime). We implement the RTV piecewise approximation.

**한국어**: 광학적으로 얇은 복사 손실 함수 $\Lambda(T)$ (erg cm³ s⁻¹)는 Rosner, Tucker & Vaiana (1978)가 구간별 멱법칙으로 근사했다. T ~ 10^5.3 K (채층/TR 정점) 근처까지 상승, 10⁶–10⁷ K에서 평탄, 그 후 고온(bremsstrahlung 영역)으로 감소한다. RTV 구간별 근사를 구현한다.

In [ ]:
def radiative_loss_rtv(T):
    """Rosner-Tucker-Vaiana (1978) piecewise power-law radiative loss function.

    Args:
        T: Temperature [K], array-like.

    Returns:
        Lambda(T) [erg cm^3 s^-1] such that optically-thin emission is n^2 Lambda(T).
    """
    T = np.asarray(T, dtype=float)
    Lam = np.zeros_like(T)
    ranges = [
        (1e4, 10**4.97, 1.09e-31, 2.0),
        (10**4.97, 10**5.67, 8.87e-17, -1.0),
        (10**5.67, 10**6.18, 1.90e-22, 0.0),
        (10**6.18, 10**6.55, 3.53e-13, -1.5),
        (10**6.55, 10**6.90, 3.46e-25, 1.0/3.0),
        (10**6.90, 10**7.63, 5.49e-16, -1.0),
        (10**7.63, 1e9, 1.96e-27, 0.5),
    ]
    for Tlo, Thi, chi, alpha in ranges:
        mask = (T >= Tlo) & (T < Thi)
        Lam[mask] = chi * T[mask] ** alpha
    return Lam


T_grid = np.logspace(4, 8, 500)
Lam = radiative_loss_rtv(T_grid)

fig, ax = plt.subplots()
ax.loglog(T_grid, Lam, 'r-', lw=2.2, label='RTV (1978) piecewise $\\Lambda(T)$')
ax.set_xlabel('T [K]')
ax.set_ylabel('$\\Lambda(T)$ [erg cm$^3$ s$^{-1}$]')
ax.set_title('Radiative Loss Function $\\Lambda(T)$ / 복사 손실 함수')
ax.legend()
plt.tight_layout()
plt.show()

for T_ex in [1e5, 1e6, 3e6, 1e7]:
    print(f'T = {T_ex:.0e} K -> Lambda = {radiative_loss_rtv(np.array([T_ex]))[0]:.3e} erg cm^3 s^-1')

## 5. Nanoflare Energy Distribution / 나노플레어 에너지 분포

**English**: Nanoflare energies are hypothesized to follow a power law $dN/dE \propto E^{-\alpha}$. Hudson (1991) showed that $\alpha > 2$ is the critical threshold: for α > 2 the total energy integral is dominated by the small-E end (i.e. nanoflares can dominate heating), while for α < 2 it is dominated by large events. We compare α = 1.5 and α = 2.5 over the observed range 10²³–10²⁹ erg (warm loops to microflares).

**한국어**: 나노플레어 에너지는 멱법칙 $dN/dE \propto E^{-\alpha}$를 따를 것으로 가정된다. Hudson (1991)은 α > 2가 임계치임을 보였다: α > 2에서는 전체 에너지 적분이 작은 E 끝에 지배되어(즉 나노플레어가 가열을 지배할 수 있음), α < 2에서는 큰 이벤트에 지배된다. 관측 범위 10²³–10²⁹ erg (warm loop ~ microflare)에서 α = 1.5와 α = 2.5를 비교한다.

In [ ]:
def nanoflare_distribution(E, alpha, N0=1.0, E_min=1e23):
    """Power-law nanoflare intensity distribution.

    Args:
        E: Energy array [erg].
        alpha: Power-law index (positive).
        N0: Normalization constant.
        E_min: Minimum energy below which distribution is zero [erg].

    Returns:
        dN/dE [events per erg].
    """
    E = np.asarray(E, dtype=float)
    return np.where(E >= E_min, N0 * (E / E_min) ** (-alpha), 0.0)


E_range = np.logspace(23, 29, 200)
dNdE_15 = nanoflare_distribution(E_range, 1.5)
dNdE_25 = nanoflare_distribution(E_range, 2.5)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].loglog(E_range, dNdE_15, 'b-', lw=2.2, label='α = 1.5 (sub-critical)')
axs[0].loglog(E_range, dNdE_25, 'r-', lw=2.2, label='α = 2.5 (super-critical)')
axs[0].set_xlabel('E [erg]')
axs[0].set_ylabel('dN/dE [events/erg]')
axs[0].set_title('Nanoflare Energy Distribution / 나노플레어 에너지 분포')
axs[0].legend()

EdN_15 = E_range * dNdE_15
EdN_25 = E_range * dNdE_25
axs[1].loglog(E_range, EdN_15, 'b-', lw=2.2, label='α=1.5: E·dN/dE ∝ $E^{-0.5}$ (big events dominate)')
axs[1].loglog(E_range, EdN_25, 'r-', lw=2.2, label='α=2.5: E·dN/dE ∝ $E^{-1.5}$ (small events dominate)')
axs[1].set_xlabel('E [erg]')
axs[1].set_ylabel('E × dN/dE [erg × events/erg]')
axs[1].set_title('Energy-weighted distribution: critical α = 2')
axs[1].legend()
plt.tight_layout()
plt.show()

total_15 = np.trapz(EdN_15, E_range)
total_25 = np.trapz(EdN_25, E_range)
print('Total integrated energy in [1e23, 1e29] erg:')
print(f'  α = 1.5: {total_15:.3e} erg (dominated by LARGE events)')
print(f'  α = 2.5: {total_25:.3e} erg (dominated by SMALL events — nanoflares can heat corona)')

## 6. Kink Mode Oscillation Period (Coronal Seismology) / kink 모드 진동 주기

**English**: A magnetic flux tube of length L, internal density ρ_i, external density ρ_e, and internal Alfvén speed v_{A,i} supports transverse kink mode oscillations with fundamental period $P_{\rm kink} = 2L/c_k$, where $c_k = v_{A,i}\sqrt{2/(1+\rho_e/\rho_i)}$. Inverting observed periods from TRACE/AIA yields estimates of the coronal magnetic field (Nakariakov et al. 1999). We map P across L and B for n = 10⁹ cm⁻³ and density contrast ρ_e/ρ_i = 0.1.

**한국어**: 길이 L, 내부 밀도 ρ_i, 외부 밀도 ρ_e, 내부 Alfvén 속도 v_{A,i}의 자기 플럭스 튜브는 기본 주기 $P_{\rm kink} = 2L/c_k$ 로 횡진동하며, $c_k = v_{A,i}\sqrt{2/(1+\rho_e/\rho_i)}$. TRACE/AIA 관측 주기로부터 코로나 자기장을 추정 (Nakariakov et al. 1999). n = 10⁹ cm⁻³, ρ_e/ρ_i = 0.1 조건에서 L과 B에 대한 P를 매핑한다.

In [ ]:
def alfven_speed(B, n):
    """Alfvén speed in fully-ionized hydrogen plasma.

    Args:
        B: Magnetic field [G].
        n: Hydrogen number density [cm^-3].

    Returns:
        v_A [cm/s] = B / sqrt(4*pi*n*m_H).
    """
    rho = n * M_H
    return B / np.sqrt(4.0 * np.pi * rho)


def kink_period(L, B, n, rho_ratio=0.1):
    """Fundamental kink mode oscillation period.

    Args:
        L: Loop length [cm].
        B: Magnetic field strength [G].
        n: Internal density [cm^-3].
        rho_ratio: External-to-internal density ratio rho_e/rho_i.

    Returns:
        P_kink [s] = 2L / c_k.
    """
    v_A = alfven_speed(B, n)
    c_k = v_A * np.sqrt(2.0 / (1.0 + rho_ratio))
    return 2.0 * L / c_k


L_values = np.array([1e9, 3e9, 1e10, 3e10])
B_range = np.linspace(3, 50, 200)
n_coronal = 1e9

fig, ax = plt.subplots()
for L in L_values:
    P = kink_period(L, B_range, n_coronal)
    ax.loglog(B_range, P, lw=2.2, label=f'L = {L:.0e} cm')
ax.axhspan(100, 1000, alpha=0.15, color='orange', label='Observed range (100–1000 s)')
ax.set_xlabel('B [G]')
ax.set_ylabel('P$_{kink}$ [s]')
ax.set_title('Kink Mode Period vs B / kink 모드 주기 vs B')
ax.legend()
plt.tight_layout()
plt.show()

P_obs, L_obs = 300.0, 1.5e10
c_k_obs = 2 * L_obs / P_obs
v_A_obs = c_k_obs * np.sqrt((1 + 0.1) / 2)
B_inferred = v_A_obs * np.sqrt(4 * np.pi * n_coronal * M_H)
print(f'Observed P = {P_obs} s, L = {L_obs:.1e} cm, n = {n_coronal:.0e} cm^-3')
print(f'  c_k  = {c_k_obs/1e5:.1f} km/s')
print(f'  v_A  = {v_A_obs/1e5:.1f} km/s')
print(f'  B    = {B_inferred:.2f} G  (inferred via coronal seismology)')

## 7. Cooling Timescales: Conductive vs Radiative / 냉각 시간 비교

**English**: For an impulsively-heated loop, conductive and radiative cooling times (Eqs. 12–13 of the review) depend differently on (T, n, L). $\tau_c \approx 1500\, n_9 L_9^2/T_{0,6}^{5/2}$ s; $\tau_r \approx 3000\, T_{M,6}^{3/2}/n_{M,9}$ s. We plot these over observed ranges to illustrate where each dominates. Conductive cooling dominates early (while T is high), radiative cooling takes over when density rises from chromospheric evaporation (Phase IV in Reale 2007).

**한국어**: 임펄시브 가열 루프에서 전도·복사 냉각 시간(리뷰 식 12–13)은 (T, n, L)에 다르게 의존한다. 관측 범위에서 이들을 그려 각각이 지배하는 영역을 보인다. 전도 냉각은 초기(T 높을 때) 지배, 복사 냉각은 채층 증발로 밀도 상승 시(Reale 2007의 Phase IV) 우위에 선다.

In [ ]:
def tau_conductive(T0, n, L):
    """Conductive cooling timescale.

    Args:
        T0: Loop apex temperature [K].
        n: Density [cm^-3].
        L: Loop length [cm].

    Returns:
        tau_c [s] = 10.5 * n * k_B * L^2 / (kappa * T0^(5/2)).
    """
    return 10.5 * n * K_B * L ** 2 / (KAPPA_SP * T0 ** 2.5)


def tau_radiative(T, n, b=1.5e-19, alpha_rad=-0.5):
    """Radiative cooling timescale using P(T) = b * T^alpha.

    Args:
        T: Temperature [K].
        n: Density [cm^-3].
        b: Loss-function prefactor [erg cm^3 s^-1 K^(-alpha)].
        alpha_rad: Power-law exponent for P(T).

    Returns:
        tau_r [s] = 3 k_B T / (n * P(T)).
    """
    return 3.0 * K_B * T / (n * b * T ** alpha_rad)


T_arr = np.logspace(5.5, 7.5, 200)
L_fix = 1e10
fig, ax = plt.subplots()
for n_fix in [1e8, 1e9, 1e10]:
    tau_c = tau_conductive(T_arr, n_fix, L_fix)
    tau_r = tau_radiative(T_arr, n_fix)
    ax.loglog(T_arr/1e6, tau_c, '--', lw=2, label=f'$\\tau_c$ (n={n_fix:.0e})')
    ax.loglog(T_arr/1e6, tau_r, '-', lw=2, label=f'$\\tau_r$ (n={n_fix:.0e})')
ax.set_xlabel('T [MK]')
ax.set_ylabel('Cooling time [s]')
ax.set_title(f'Cooling Timescales vs T for L = {L_fix:.0e} cm')
ax.legend()
plt.tight_layout()
plt.show()

print('Typical AR loop (T=3e6 K, n=1e9 cm^-3, L=1e10 cm):')
print(f'  tau_c = {tau_conductive(3e6, 1e9, 1e10):.1f} s')
print(f'  tau_r = {tau_radiative(3e6, 1e9):.1f} s')

## 8. Summary / 요약

**English**: This notebook reproduced the principal quantitative results of Reale (2014):

1. **RTV scaling** — T_max ∝ (pL)^(1/3), consistent with hot/warm/flaring parameter bands of Table 1 of the review.
2. **Hydrostatic profile** — Steep transition region and gradual coronal plateau for AR vs Empty loops (Fig. 14 of the review).
3. **Spitzer conductivity** — T^{5/2} scaling dominates at coronal temperatures.
4. **Radiative Λ(T)** — RTV piecewise approximation showing peak near T ~ 10^5 K and plateau at MK temperatures.
5. **Nanoflare power law** — α = 1.5 vs 2.5 crossing the critical α = 2 threshold for coronal heating (Hudson 1991).
6. **Kink seismology** — Inverted TRACE-like oscillation parameters to infer B ≈ 10–30 G in the corona.
7. **Cooling timescales** — Conductive dominates when T high, radiative dominates at moderate T and high n.

**한국어**: 이 노트북은 Reale (2014)의 주요 정량 결과를 재현했다:

1. **RTV 스케일링** — T_max ∝ (pL)^(1/3), 리뷰 Table 1의 hot/warm/flaring 파라미터 영역과 일치.
2. **정역학 단면** — AR과 Empty 루프에 대한 가파른 전이영역과 점진적 코로나 평탄부 (리뷰 Fig. 14).
3. **Spitzer 전도도** — 코로나 온도에서 T^{5/2} 스케일링이 지배적.
4. **복사 Λ(T)** — T ~ 10^5 K 정점과 MK 온도 평탄부를 보이는 RTV 구간별 근사.
5. **나노플레어 멱법칙** — 코로나 가열의 임계 α = 2 임계치를 가로지르는 α = 1.5 vs 2.5 (Hudson 1991).
6. **Kink 지진학** — TRACE형 진동 파라미터를 역변환하여 코로나 B ≈ 10–30 G 추정.
7. **냉각 시간** — T 높을 때 전도, 중간 T와 높은 n에서 복사가 지배.